# Paso 7 — Extensión experimental: discretización de las variables

**Condición que se modifica:** la forma en que el modelo ve las variables. En lugar de valores continuos (originales o derivadas), cada variable se reemplaza por indicadores de rango (*k* rangos por variable). Todo lo demás se mantiene igual: los datos simulados, las particiones aleatorias 80/20 (mismas semillas que el Paso 5), los cuatro clasificadores y sus hiperparámetros.

**Motivación.** El trabajo de grado estudia un esquema de *scoring* en el que el modelo recibe rangos atestados en lugar de valores exactos (pruebas de conocimiento cero). La pregunta es cuánto desempeño se pierde cuando la información de entrada se reduce a rangos.

**Hipótesis previas** (formuladas antes de ver los resultados):
1. El desempeño disminuye al reducir el número de rangos *k*; con *k* = 2 la caída es la mayor.
2. Con *k* moderado (5 a 10) el desempeño se mantiene cerca del obtenido con variables derivadas.

**Comparación:** variables originales y derivadas (Paso 5) frente a variables discretizadas con *k* = 2, 3, 5 y 10.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from features import FeatureBuilder
from evaluate import correr_extension

TARGET = "SeriousDlqin2yrs"
(RAIZ / "results").mkdir(exist_ok=True); (RAIZ / "figures").mkdir(exist_ok=True)

## 7.1 Configuración
`N_RUNS` es el número de corridas por valor de *k*; deben ser como máximo las corridas ya disponibles en el Paso 5 (`results/paso5_resultados_por_corrida.csv`), porque la comparación usa las mismas particiones. `MUESTRA` permite usar una submuestra en pruebas rápidas (`None` usa todos los registros).

In [ ]:
N_RUNS = 3
KS = [2, 3, 5, 10]
MUESTRA = None

df = pd.read_csv(RAIZ / "data" / "datos_simulados.csv")
if MUESTRA:
    df = df.sample(MUESTRA, random_state=0)
paso5 = pd.read_csv(RAIZ / "results" / "paso5_resultados_por_corrida.csv")
print("corridas disponibles en el Paso 5:", paso5["run"].nunique(), "| corridas de la extensión:", N_RUNS)
assert N_RUNS <= paso5["run"].nunique(), "El Paso 5 tiene menos corridas que N_RUNS"
df.shape

## 7.2 Cortes de los rangos
Los cortes se calculan solo con el conjunto de entrenamiento: cuantiles para las variables continuas y, en las variables de conteo con mucha masa en el mínimo (por ejemplo, atrasos), los primeros valores distintos (0 | 1 | 2 | ...). Si una variable tiene valores faltantes en entrenamiento, se agrega un rango adicional "NA". Ejemplo con *k* = 5 sobre la partición de la corrida 0:

In [ ]:
train0, _ = train_test_split(df, test_size=0.2, random_state=1000)
fb = FeatureBuilder("discretized", n_bins=5).fit(train0)
cortes = pd.DataFrame({
    "variable": list(fb.cortes_.keys()),
    "cortes (limite inferior de cada rango, excepto el primero)": [[round(float(x), 2) for x in q] for q in fb.cortes_.values()],
    "rango NA": [fb.tiene_na_[c] for c in fb.cortes_],
})
cortes.to_csv(RAIZ / "results" / "paso7_cortes_k5.csv", index=False)
print("columnas de entrada por valor de k:", {k: len(FeatureBuilder("discretized", n_bins=k).fit(train0).columnas_) for k in KS})
cortes

## 7.3 Experimento
Se ejecutan los cuatro clasificadores con variables discretizadas para cada *k* y cada corrida. Los resultados se guardan de forma incremental en `results/paso7_resultados_por_corrida.csv`; si la ejecución se interrumpe, se retoma donde quedó.

In [ ]:
ext = correr_extension(df, n_runs=N_RUNS, ks=KS, destino=RAIZ / "results" / "paso7_resultados_por_corrida.csv")
ext.shape

## 7.4 Comparación con variables originales y derivadas

In [ ]:
base = paso5[paso5["run"] < N_RUNS]
todo = pd.concat([base, ext[ext["run"] < N_RUNS]], ignore_index=True)
orden_vars = ["original", "derived"] + [f"discretized_k{k}" for k in KS]
orden_clf = ["LLRE", "RF", "LLR", "CART"]

resumen = (todo.groupby(["clasificador", "variables"])[["auc", "f1"]].agg(["mean", "std"]).round(4))
resumen.columns = [f"{m}_{s}" for m, s in resumen.columns]
resumen.to_csv(RAIZ / "results" / "paso7_resumen.csv")

auc = todo.pivot_table(index="clasificador", columns="variables", values="auc").loc[orden_clf, orden_vars].round(4)
f1 = todo.pivot_table(index="clasificador", columns="variables", values="f1").loc[orden_clf, orden_vars].round(4)
print("AUC medio"); display(auc)
print("F-measure medio"); display(f1)

In [ ]:
# Variación respecto a las variables derivadas (mismas particiones)
filas = []
for clf in orden_clf:
    for k in KS:
        v = f"discretized_k{k}"
        a_d = auc.loc[clf, "derived"]; f_d = f1.loc[clf, "derived"]
        filas.append({"clasificador": clf, "k": k,
                      "AUC": auc.loc[clf, v], "dif_AUC_vs_derived": round(auc.loc[clf, v] - a_d, 4),
                      "F1": f1.loc[clf, v], "dif_F1_vs_derived": round(f1.loc[clf, v] - f_d, 4)})
variacion = pd.DataFrame(filas)
variacion.to_csv(RAIZ / "results" / "paso7_variacion_vs_derivadas.csv", index=False)
variacion

In [ ]:
colores = {"LLRE": "#1b6ca8", "RF": "#d95f02", "LLR": "#2a9d8f", "CART": "#7a5195"}
pos = np.arange(len(KS))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, tabla, tit in zip(axes, [auc, f1], ["AUC", "F-measure"]):
    for clf in orden_clf:
        y = [tabla.loc[clf, f"discretized_k{k}"] for k in KS]
        ax.plot(pos, y, marker="o", color=colores[clf], label=clf)
        ax.axhline(tabla.loc[clf, "derived"], color=colores[clf], ls="--", lw=0.9, alpha=0.7)
    ax.set_xticks(pos); ax.set_xticklabels(KS)
    ax.set_xlabel("número de rangos por variable (k)"); ax.set_ylabel(tit); ax.set_title(tit)
    ax.grid(alpha=0.25)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False,
           title="línea continua: variables discretizadas  |  línea punteada: variables derivadas (Paso 5)")
fig.suptitle(f"Efecto de la discretización (promedio de {N_RUNS} corrida(s))")
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.savefig(RAIZ / "figures" / "paso7_discretizacion.png", dpi=130)